<a href="https://colab.research.google.com/github/genji970/LLM_pretraining/blob/main/pretraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install transformers

# **data preparation**

In [ ]:
from dataclasses import dataclass
import string , random

@dataclass
class Data_Config:
  data_path : str = None
  # task of data / purpose of data
  data_kind : str = None
  # Number of data
  sample_num : int = None
  # How many data will be loaded in Dataset
  chunk : int = None
  name : str = None

  token_key : str = None
  token_id : int = None

  data_structure : tuple | None = ("question" , "reference" , "answer")

  # what is field()?

# **processing tokenizer**

In [ ]:
import re
import time
from typing import List , Dict

class Tokenizer:
  def __init__(self , data_name : str = None , dictionary : dict[str , int] = {}):
    self.data_name = data_name
    self.dictionary = dictionary
    self.max_count = int(len(self.dictionary)) if self.dictionary is not None else 0

  # tokenize -> encoder
  def tokenize(self,num_samples : list[str | None] = None) -> list[str | None]:

    # need parallel since it is one by one and take long time and it's inefficient
    tokens_list = []
    for text in num_samples:

      # #, ##, ### 제거
      text = re.sub(r"#+", " ", text)

      # 영어 단어와 문장부호를 각각 추출
      raw_tokens = re.findall(
          r"[A-Za-z]+|[.,!?;:()]",
          text,
      )

      tokens = []

      for token in raw_tokens:
        match = re.fullmatch(
          r"([A-Za-z]+?)(ing|ed)",
          token,
          flags=re.IGNORECASE,
        )

        if match:
          tokens.append(match.group(1))
          tokens.append(match.group(2))
        else:
          tokens.append(token)
      tokens_list.extend(tokens)
    return tokens_list

  def tokenizer_encoding(self , text_list : list[str | None]) -> Dict:

    for sample in text_list:
      # since dict is hash table, its O(1) in searching
      if sample not in self.dictionary:
        self.max_count += 1
        self.dictionary[sample] = int(self.max_count)
    return self.dictionary

  def inserting_special_token(self, special_token : str | None = None):
    assert type(special_token) == str, "Warning! special_token you added is not str."
    if isinstance(special_token , str):
      if special_token not in self.dictionary.keys():
        self.dictionary[special_token] = self.max_count + 1
        self.max_count += 1

  def print_max_num_encoder_dictionary(self):
    print(f"whole number of tokens/index of dictionary : {self.max_count}")

Might be good to mapping natural language to index with semantic criteria

In [ ]:
dictionary = {'son' : 1, 'daughter' : 2}
print(len(dictionary))
tokenizer = Tokenizer(dictionary=dictionary)
tokens=tokenizer.tokenize(['I want to go to school but yesterday there was ##.','I want to go to school but yesterday there was ##.'])
dic_token=tokenizer.tokenizer_encoding(tokens)
print(dic_token)
tokenizer.print_max_num_encoder_dictionary()
type(dic_token['there'])

2
{'son': 1, 'daughter': 2, 'I': 3, 'want': 4, 'to': 5, 'go': 6, 'school': 7, 'but': 8, 'yesterday': 9, 'there': 10, 'was': 11, '.': 12}
whole number of tokens/index of dictionary : 12


int

# **processing positional embedding**

q, k: [batch_size, num_heads, sequence_length, head_dim]

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision , torchaudio

class RoPE_Positional_Embedding(nn.Module):
  def __init__(self , batch_num, embed_dim, context_length , head_dim):
    super().__init__()
    self.batch_num = batch_num
    self.embed_dim = embed_dim
    self.context_length = context_length
    self.head_dim = head_dim

    if head_dim % 2 != 0:
      raise ValueError("head_dim should be even")

    if embed_dim % head_dim != 0:
      raise ValueError("embed_dim should be divided by head_dim")

    p = torch.arange(0, context_length , 1, dtype=torch.float32)
    p=p[None,None,:,None]

    i=torch.arange(0, head_dim , 2, dtype = torch.float32)

    tmp_frequency = 10000 ** (((-1) * i) / head_dim)

    tmp_frequency=tmp_frequency[None,None,None,:]

    # frequency *=p , frequency might not change its shape so it could make error. need to use new variable.
    frequency = (tmp_frequency * p).repeat(int(batch_num), int(embed_dim//head_dim),int(1),int(1))

    self.register_buffer(
      "frequency",
      frequency,
      persistent=False,
    )

  def transformation(self):
    return torch.sin(self.frequency) , torch.cos(self.frequency)

  def construct_embedding(self,x, sin, cos):
    x_even=x[...,0::2]
    x_odd=x[...,1::2]

    rotate_even=x_even * cos - x_odd * sin
    rotate_odd=x_even * cos + x_odd * sin

    return torch.stack([rotate_even,rotate_odd],dim=-1).flatten(-2)

In [ ]:
import torch
batch_num=4
embed_dim=32
head_dim = 16
context_length = 4

"""
p = torch.arange(0, context_length , 1, dtype=torch.float32)
p=p[None,None,:,None]

i=torch.arange(0, head_dim , 2, dtype = torch.float32)

frequency = 10000 ** (((-1) * i) / head_dim)
frequency=frequency[None,None,None,:]
new=p*frequency
"""

positional_embedding=RoPE_Positional_Embedding(batch_num,embed_dim,context_length,head_dim)
positional_embedding_0, positional_embedding_1 = positional_embedding.transformation()
print(positional_embedding_0.shape , positional_embedding_1.shape)

torch.Size([4, 2, 4, 8]) torch.Size([4, 2, 4, 8])


# **Constructing Attention**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision, torchaudio

# **self.attention block**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision, torchaudio

class Self_Attention(nn.Module):
  """
  If need causal masking -> do .causal_masking(attention_matrix) <- method

  """
  def __init__(self, batch_num , embed_dim, context_length, num_head , position_flag : int , causal_process_flag : bool):
    super().__init__()
    self.batch_num = batch_num
    self.embed_dim = embed_dim
    self.context_length=context_length
    self.num_head = num_head
    self.head_dim = embed_dim // num_head
    self.position_flag = position_flag # 0 : RoPE, 1 : normal positional embedding
    self.causal_process_flag = causal_process_flag # if causal then True

    if embed_dim % num_head != 0:
      raise ValueError("embed_dim % num_head should be zero")

    self.positional_embedding = RoPE_Positional_Embedding(batch_num, embed_dim, context_length , embed_dim//num_head)

    self.query_embedding =nn.Parameter(torch.randn(self.embed_dim, self.embed_dim)) # Normal distribution
    self.key_embedding = nn.Parameter(torch.randn(self.embed_dim, self.embed_dim))
    self.value_embedding = nn.Parameter(torch.randn(self.embed_dim, self.embed_dim))

  def construct_q_k_v(self, input):
    return input@self.query_embedding ,input@self.key_embedding,input@self.value_embedding

  def causal_masking(self, attention_matrix):

    if attention_matrix.shape != torch.Size([self.batch_num, self.num_head, self.context_length, self.context_length]):
      raise ValueError("attention_matrix.shape is wrong. Should fix this!")
    # attention_matrix.shape : [B,H,C,C] before multiplying value embedding
    mask =torch.triu(
        torch.ones(
            self.context_length , self.context_length, \
            device = attention_matrix.device,
            dtype = torch.bool),
        diagonal=1
        )

    return attention_matrix.masked_fill(mask[None,None,:,:],-torch.inf)

  # not constructed
  def normal_positional_attention(self, query, key, value):
    """
    torch.sqrt(torch.tensor(self.head_dim)) is made in cpu tensor. error might occur.
    """
    pre_attention_weight = (query @ key.transpose(-1,-2)) / torch.sqrt(torch.tensor(self.head_dim))
    attention_score=(torch.softmax(pre_attention_weight , dim=-1)) @ value
    return attention_score

  def rope_positional_attention(self, query, key, value , causal_process_flag : bool):
    sin_transformation,cos_transformation=self.positional_embedding.transformation()
    query=self.positional_embedding.construct_embedding(query,sin_transformation,cos_transformation)
    key=self.positional_embedding.construct_embedding(key,sin_transformation,cos_transformation)

    pre_attention_weight = (query @ key.transpose(-1,-2)) / (self.head_dim**0.5)

    if causal_process_flag:
      causal_pre_attention_weight=self.causal_masking(pre_attention_weight)
      return (torch.softmax(causal_pre_attention_weight , dim=-1)) @ value

    return torch.softmax(pre_attention_weight , dim=-1) @ value

  # construct_q_k_v -> attention -> forward
  def forward(self , x): # x is just input

    if x.shape != torch.Size([self.batch_num, self.context_length, self.embed_dim]):
      raise ValueError("input data's shape is not what this model wants")

    if self.position_flag == 0:
      query,key,value=self.construct_q_k_v(x)
      query=query.view(self.batch_num, self.context_length , self.num_head, self.embed_dim//self.num_head).transpose(1,2).contiguous()
      key=key.view(self.batch_num, self.context_length , self.num_head, self.embed_dim//self.num_head).transpose(1,2).contiguous()
      value=value.view(self.batch_num, self.context_length , self.num_head, self.embed_dim//self.num_head).transpose(1,2).contiguous()

      if query.shape != torch.Size([self.batch_num, self.num_head, self.context_length, self.head_dim]) or \
        key.shape != torch.Size([self.batch_num, self.num_head, self.context_length, self.head_dim]) or \
        value.shape != torch.Size([self.batch_num, self.num_head, self.context_length, self.head_dim]):
        raise ValueError("query.shape and key.shape and value.shape must be equal to [batch_num,num_head, context_length , head_dim]")

      attention_score=self.rope_positional_attention(query,key,value,self.causal_process_flag)
    else:
      raise ValueError("normal positional embedding is not yet built.")

    assert self.position_flag == 0 or 1, "Warning! self.position_flag should be either 0/1."

    return attention_score.transpose(2,1).reshape(self.batch_num,self.context_length,self.embed_dim)

In [ ]:
mask=torch.triu(torch.ones(self.batch_num , self.num_head, self.context_length , self.context_length, \
                device = #class_name#.device,
                dtype = torch.bool)

                , diagonal=1)

mask=mask.masked_fill(
    mask,
    -torch.inf
)


SyntaxError: invalid syntax (2466697702.py, line 3)

In [ ]:
batch_num = 2
embed_dim = 8
context_length= 16
num_head= 4
position_flag= 0
causal_process_flag= False
x = torch.rand(batch_num, context_length, embed_dim)

self_attention=Self_Attention(batch_num, embed_dim, context_length, num_head, position_flag , causal_process_flag)

print(f"x : {x.shape}\n", sep="")
print(f"attention : {self_attention(x).shape}")

x : torch.Size([2, 16, 8])

attention : torch.Size([2, 16, 8])


In [ ]:
torch.tensor([[1,2],[3,4]]).shape == torch.tensor([2,2])

False

# **transformer block building**

In [ ]:
class Transformer(nn.Module):

  def __init__(self, batch_num : int, embed_dim : int, context_length : int, num_head : int, position_flag : int, causal_process_flag : bool,vocab_dim : int, crossentropy_flag : bool , is_transformer_block_is_final : bool):
    super().__init__()
    self.batch_num=batch_num
    self.embed_dim=embed_dim
    self.context_length=context_length
    self.num_head=num_head
    self.position_flag=position_flag
    self.causal_process_flag=causal_process_flag
    self.is_transformer_block_is_final = is_transformer_block_is_final

    if self.is_transformer_block_is_final:
      self.proj = nn.Linear(embed_dim, vocab_dim)
      self.softmax = nn.Softmax(dim=-1)
      self.layernorm=nn.LayerNorm(embed_dim)

    self.feed_forward = nn.Sequential(
    nn.Linear(embed_dim, embed_dim * 4),
    nn.GELU(),
    nn.Linear(embed_dim * 4, embed_dim),
    nn.Dropout(0.1),
    )

    self.layernorm_list=nn.ModuleList([nn.LayerNorm(embed_dim) for _ in range(2)])

    self.crossentropy_flag = crossentropy_flag

    self.causal_attention=Self_Attention(batch_num, embed_dim , context_length, num_head, position_flag, 1) # if causal_process_flag==1, it is causal_attention_process


  def forward(self, x):

    normalized_x=self.layernorm_list[0](x)
    z=self.causal_attention(normalized_x)
    z+=x
    pre_z=z
    z=self.layernorm_list[1](z)
    z=self.feed_forward(z)+pre_z

    if self.is_transformer_block_is_final:
      z=self.layernorm(z)
      logit=self.proj(z)
      vocab_prob=self.softmax(logit)
      return vocab_prob if not self.crossentropy_flag else logit
    else:
      return z

# **stacking transformer block **

In [ ]:
batch_num=2
embed_dim=32
context_length=16
num_head=4
position_flag=0
causal_process_flag=True
vocab_dim=128
crossentropy_flag=0
is_transformer_block_is_final=0

class Stacking(nn.Module):

  def __init__(self, block_num : int, batch_num : int, embed_dim : int, context_length : int, num_head:int, position_flag, causal_process_flag, vocab_dim, crossentropy_flag,is_transformer_block_is_final):
    super().__init__()
    self.block_num = block_num
    self.batch_num=batch_num
    self.embed_dim=embed_dim
    self.context_length=context_length
    self.num_head=num_head
    self.position_flag=position_flag
    self.causal_process_flag=causal_process_flag
    self.vocab_dim=vocab_dim
    self.crossentropy_flag=crossentropy_flag
    self.is_transformer_block_is_final=is_transformer_block_is_final

    self.transformer_block_list = nn.ModuleList([Transformer(batch_num, \
            embed_dim, \
            context_length, \
            num_head, \
            position_flag, \
            causal_process_flag, \
            vocab_dim, \
            crossentropy_flag, \
            is_transformer_block_is_final=False
            ) for _ in range(block_num-1)])

    self.final_block = Transformer(batch_num, \
            embed_dim, \
            context_length, \
            num_head, \
            position_flag, \
            causal_process_flag, \
            vocab_dim, \
            crossentropy_flag, \
            is_transformer_block_is_final=True
            )
  def forward(self, x):
    for transformer_block in self.transformer_block_list:
      x=transformer_block(x)
    return self.final_block(x)

# **train pipeline building**

In [ ]:

config = [
    'output_dir' : ,
    'learning_rate' : 2e-5,
    'per_device_train_batch_size' : 16,
    'per_device_eval_batch_size' : 16,
    'num_train_epoch' : 2,
    'weight_decay' : ,    # scheduling strategy is from deepmind paper.
    'eval_strategy' : 'epoch',
    'save_strategy' : 'epoch',
    'push_to_hub' : True,
    'load_checkpoint' : False,
    'load_specific_checkpoint' : False,
    'block_num' : 8, # number of transformer
    'batch_num' : 2,
    'embed_dim' : 512,
    'context_length' : 256,
    'num_head' : 8,
    'position_flag' : 0,  # 0 : RoPE , 1: normal positional embedding
    'causal_process_flag' : True, # decoder only version
    'vocab_dim': 16384,
    'crossentropy_flag': False, # not using cross entropy
    'is_transformer_block_is_final': False, # Whether or not to use projection and softmax to output z

]

In [ ]:
!pip install -U huggingface_hub accelerate datasets trl bitsandbytes

need huggingface token, and login using hub / enroll in os.enviorn

In [ ]:
from transformers import Trainer TrainingArguments
import accelerate # for distributed environment

training_args = TrainingArguments(
    output_dir=config['output_dir'],
    learning_rate=config['learning_rate'],
    per_device_train_batch_size=config['per_device_train_batch_size'],
    per_device_eval_batch_size=config['per_device_eval_batch_size'],
    num_train_epochs=config['num_train_epochs'],
    weight_decay=config['weight_decay'],        # scheduling strategy is from deepmind paper.
    eval_strategy=config['eval_strategy'],
    save_strategy=config['save_strategy'],
    load_best_model_at_end=True,
    push_to_hub=config['push_to_hub'],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

if config['load_checkpoint'] == False:
  trainer.train()
else:
  # 최신 체크포인트에서 재개 restart from recent checkpoint
  if config['load_specific_checkpoint'] == True:
    # 출력 디렉토리에 저장된 특정 체크포인트에서 재개 restart from specific checkpoint from output directory
    trainer.train(resume_from_checkpoint="your-model/checkpoint-1000")
  else:
    trainer.train(resume_from_checkpoint=True)